# QA Generator - NL to Cypher Dataset

Generate pasangan (pertanyaan NL, Cypher query) untuk fine-tuning query model.

**Mengikuti pola dari `qa_generator.ipynb` reference.**

**Requirements:**
1. Python >= 3.10
2. Neo4j running di `bolt://localhost:7687`
3. Google Sheets API credentials di `.env`
4. Gemini API key di `.env`

**How to get Google API key:**
1. Go to https://console.cloud.google.com/
2. Enable Google Sheets API
3. Create service account credentials
4. Copy `GOOGLE_SHEETS_CLIENT_EMAIL` and `GOOGLE_SHEETS_PRIVATE_KEY` to `.env`

## Data Preparation

In [ ]:
# Google Authentication & Config
import os, sys, json
import pandas as pd
from typing import Dict, List
from dotenv import load_dotenv

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

load_dotenv(os.path.join(ROOT_DIR, '.env'), override=True)

# --- Config ---
# TODO: Replace with your actual Google Spreadsheet ID
GOOGLE_SPREADSHEET_ID: str = os.getenv('GOOGLE_SPREADSHEET_ID', 'YOUR_SPREADSHEET_ID_HERE')
GOOGLE_SPREADSHEET_URL: str = f"https://docs.google.com/spreadsheets/d/{GOOGLE_SPREADSHEET_ID}/edit?usp=sharing"

GEMINI_API_KEY: str = os.getenv('GEMINI_API_KEY', '')
GOOGLE_SHEETS_CLIENT_EMAIL: str = os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
GOOGLE_SHEETS_PRIVATE_KEY: str = os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', '')

print(f"Root: {ROOT_DIR}")
print(f"Spreadsheet: {GOOGLE_SPREADSHEET_ID[:20]}...")
print(f"Gemini API: {'set' if GEMINI_API_KEY else 'NOT SET'}")
print(f"GSheets Email: {GOOGLE_SHEETS_CLIENT_EMAIL or 'NOT SET'}")

In [ ]:
# Connect to Neo4j
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'passwd123')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    stats = session.run("""
        MATCH (n) 
        RETURN labels(n)[0] AS type, count(n) AS cnt 
        ORDER BY cnt DESC
    """).data()

print("Neo4j KG stats:")
for s in stats:
    print(f"  {s['type']}: {s['cnt']}")

### Load seed data from KG

In [ ]:
# Gather entity examples from Neo4j for template filling
entity_queries = {
    "Pasal": "MATCH (p:Pasal) RETURN p.label AS label, substring(p.content, 0, 200) AS content LIMIT 15",
    "Bab": "MATCH (b:Bab) RETURN b.label AS label, b.content AS content LIMIT 10",
    "EntitasHukum": "MATCH (e:EntitasHukum) RETURN e.label AS label, e.content AS content LIMIT 15",
    "PerbuatanHukum": "MATCH (ph:PerbuatanHukum) RETURN ph.label AS label, ph.content AS content LIMIT 15",
    "Sanksi": "MATCH (s:Sanksi) RETURN s.label AS label, s.content AS content LIMIT 15",
    "KonsepHukum": "MATCH (k:KonsepHukum) RETURN k.label AS label, k.content AS content LIMIT 15",
}

entity_data = {}
with driver.session() as session:
    for node_type, query in entity_queries.items():
        results = session.run(query).data()
        entity_data[node_type] = results
        print(f"{node_type}: {len(results)} entities loaded")

# Preview
for ntype, entities in entity_data.items():
    if entities:
        print(f"\n{ntype} examples:")
        for e in entities[:3]:
            print(f"  - {e['label']}")

### Step 1: Template-based generation

In [ ]:
from finetuning.query_model.generate_training_data import (
    generate_from_templates, validate_cypher_queries, KG_SCHEMA,
    SYSTEM_INSTRUCTION, TrainingSample
)

template_samples = generate_from_templates(driver)

# Category breakdown
categories = {}
for s in template_samples:
    categories[s.category] = categories.get(s.category, 0) + 1

print(f"Total template-based pairs: {len(template_samples)}")
for cat, cnt in sorted(categories.items()):
    print(f"  {cat}: {cnt}")

In [ ]:
# Preview samples per category
import random

for cat in sorted(categories.keys()):
    cat_samples = [s for s in template_samples if s.category == cat]
    sample = random.choice(cat_samples)
    print(f"[{cat}]")
    print(f"  Q: {sample.question}")
    print(f"  C: {sample.response[:120]}...")
    print()

### Synthetics data prompt

Prompt template untuk LLM-assisted generation — bisa di-load dari Google Sheets atau hardcoded.

In [ ]:
# Synthetics data prompt (hardcoded — can be loaded from Google Sheets later)
from finetuning.query_model.generate_training_data import (
    LLM_GENERATION_PROMPT_SYSTEM, LLM_GENERATION_PROMPT_USER
)

print("SYSTEM PROMPT (first 500 chars):")
print(LLM_GENERATION_PROMPT_SYSTEM[:500])
print("\n...")
print(f"\nTotal system prompt length: {len(LLM_GENERATION_PROMPT_SYSTEM)} chars")

## Generate QA pairs (LLM-assisted)

In [ ]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

print(f"Model ready: gemini-2.5-flash")

### Sanity check

In [ ]:
# Sanity check: generate 3 pairs
example_entities_str = json.dumps(
    {k: [e['label'] for e in v[:5]] for k, v in entity_data.items()},
    ensure_ascii=False, indent=2
)
existing_qs = "\n".join([f"- {s.question}" for s in template_samples[:10]])

user_prompt = LLM_GENERATION_PROMPT_USER.format(
    total_data=3,
    example_entities=example_entities_str,
    existing_questions=existing_qs,
)

response = model.generate_content(
    [LLM_GENERATION_PROMPT_SYSTEM, user_prompt],
    generation_config={'response_mime_type': 'application/json', 'temperature': 0.9}
)

result = json.loads(response.text)
print(json.dumps(result, indent=4, ensure_ascii=False))

### Batch generate

In [ ]:
from finetuning.query_model.generate_training_data import generate_with_llm

NUM_LLM_SAMPLES = 50  # Adjust as needed

llm_samples = generate_with_llm(
    driver, template_samples, NUM_LLM_SAMPLES, GEMINI_API_KEY
)
print(f"\nLLM-assisted: {len(llm_samples)} valid pairs generated")

## Validate & Save

In [ ]:
# Combine and validate
all_samples = template_samples + llm_samples

valid_samples, errors = validate_cypher_queries(all_samples, driver)
print(f"Total: {len(all_samples)} -> Valid: {len(valid_samples)}, Invalid: {len(errors)}")

if errors:
    print(f"\nFirst 3 errors:")
    for err in errors[:3]:
        print(f"  Q: {err['question'][:60]}")
        print(f"  E: {err['error'][:80]}")
        print()

In [ ]:
# Save to CSV (local fallback)
from finetuning.query_model.generate_training_data import (
    save_to_csv, save_prompt_template_csv
)

OUTPUT_DIR = os.path.join(ROOT_DIR, 'finetuning', 'query_model', 'data')

train_path, val_path = save_to_csv(valid_samples, OUTPUT_DIR)
prompt_path = save_prompt_template_csv(OUTPUT_DIR)

print(f"Train: {train_path}")
print(f"Val:   {val_path}")
print(f"Prompt: {prompt_path}")

In [ ]:
# Final stats
import csv

for name, path in [('Train', train_path), ('Val', val_path)]:
    with open(path, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    cats = {}
    for r in rows:
        cats[r['category']] = cats.get(r['category'], 0) + 1
    print(f"{name}: {len(rows)} samples")
    for cat, cnt in sorted(cats.items()):
        print(f"  {cat}: {cnt}")
    print()

## (Optional) Upload to Google Sheets

In [ ]:
# Upload to Google Sheets (skip if credentials not configured)
if GOOGLE_SHEETS_CLIENT_EMAIL and GOOGLE_SPREADSHEET_ID != 'YOUR_SPREADSHEET_ID_HERE':
    from finetuning.query_model.generate_training_data import upload_to_google_sheets
    upload_to_google_sheets(valid_samples, GOOGLE_SPREADSHEET_ID)
else:
    print("Google Sheets not configured. Data saved to CSV only.")
    print(f"To upload later, set GOOGLE_SPREADSHEET_ID and credentials in .env")

In [ ]:
driver.close()
print("Done!")